In [38]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle




df = pd.read_csv('car_prediction_data.csv')
print(df.head())
print(df.isnull().sum())
print(df.describe())

  Car_Name  Year  Selling_Price  Present_Price  Kms_Driven Fuel_Type  \
0     ritz  2014           3.35           5.59       27000    Petrol   
1      sx4  2013           4.75           9.54       43000    Diesel   
2     ciaz  2017           7.25           9.85        6900    Petrol   
3  wagon r  2011           2.85           4.15        5200    Petrol   
4    swift  2014           4.60           6.87       42450    Diesel   

  Seller_Type Transmission  Owner  
0      Dealer       Manual      0  
1      Dealer       Manual      0  
2      Dealer       Manual      0  
3      Dealer       Manual      0  
4      Dealer       Manual      0  
Car_Name         0
Year             0
Selling_Price    0
Present_Price    0
Kms_Driven       0
Fuel_Type        0
Seller_Type      0
Transmission     0
Owner            0
dtype: int64
              Year  Selling_Price  Present_Price     Kms_Driven       Owner
count   301.000000     301.000000     301.000000     301.000000  301.000000
mean   2013.627

In [39]:
df = df.drop('Car_Name', axis=1)
df['Car_Age'] = 2025 - df['Year']
df = df.drop('Year', axis=1)

df['Fuel_Type'] = df['Fuel_Type'].map({'Petrol': 0, 'Diesel': 1, 'CNG': 2})
df['Seller_Type'] = df['Seller_Type'].map({'Dealer': 0, 'Individual': 1})
df['Transmission'] = df['Transmission'].map({'Manual': 0, 'Automatic': 1})

df['Price_per_age'] = df['Present_Price'] / df['Car_Age']
df['Kms_per_age'] = df['Kms_Driven'] / df['Car_Age']

print(df.head())
print(df.dtypes)

   Selling_Price  Present_Price  Kms_Driven  Fuel_Type  Seller_Type  \
0           3.35           5.59       27000          0            0   
1           4.75           9.54       43000          1            0   
2           7.25           9.85        6900          0            0   
3           2.85           4.15        5200          0            0   
4           4.60           6.87       42450          1            0   

   Transmission  Owner  Car_Age  Price_per_age  Kms_per_age  
0             0      0       11       0.508182  2454.545455  
1             0      0       12       0.795000  3583.333333  
2             0      0        8       1.231250   862.500000  
3             0      0       14       0.296429   371.428571  
4             0      0       11       0.624545  3859.090909  
Selling_Price    float64
Present_Price    float64
Kms_Driven         int64
Fuel_Type          int64
Seller_Type        int64
Transmission       int64
Owner              int64
Car_Age            int64
P

In [40]:
X= df.drop('Selling_Price', axis=1).values
y= df['Selling_Price'].values


split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)



X_train shape: (240, 9)
X_test shape: (61, 9)


In [41]:
X_mean = np.mean(X_train, axis=0)
X_std  = np.std(X_train, axis=0)

X_train_scaled = (X_train - X_mean) / X_std
X_test_scaled  = (X_test  - X_mean) / X_std 

y_mean = np.mean(y_train)
y_std  = np.std(y_train)

y_train_scaled = (y_train - y_mean) / y_std
y_test_scaled  = (y_test  - y_mean) / y_std

In [42]:
def predict(X, w, b):
    return np.dot(X, w) + b

def mse(y_true,y_pred):
    return np.mean((y_true - y_pred) ** 2)

def mae(y_true,y_pred):
    return np.abs(y_true - y_pred).mean() 

def compute_gradient(X, y, w, b):
    y_pred = predict(X, w, b)
    error = y - y_pred
    dw = np.mean(-2 * X * error.reshape(-1, 1), axis=0)
    db = np.mean(-2 * error)
    return dw, db

w = np.zeros(X_train_scaled.shape[1])
b = 0.0
learning_rate = 0.05
epochs = 1000
loss_history = []

for epoch in range(epochs):
    predictions = predict(X_train_scaled, w, b)
    dw, db = compute_gradient(X_train_scaled, y_train_scaled, w, b)
    w -= learning_rate * dw
    b -= learning_rate * db
    current_mse = mse(y_train_scaled, predictions)
    loss_history.append(current_mse)
    if epoch % 20 == 0:
     print(f"Epoch {epoch} → w={np.round(w,4)} b={b:.4f} MSE={current_mse:.1f}")

Epoch 0 → w=[ 0.0884  0.0052  0.0547 -0.0553  0.041  -0.0082 -0.0193  0.0975  0.0148] b=0.0000 MSE=1.0
Epoch 20 → w=[ 0.3273 -0.0421  0.1284 -0.1041  0.0986 -0.029  -0.125   0.4454 -0.0126] b=0.0000 MSE=0.1
Epoch 40 → w=[ 0.3111 -0.0346  0.1054 -0.0669  0.0698 -0.03   -0.1264  0.5194 -0.0115] b=0.0000 MSE=0.1
Epoch 60 → w=[ 0.2784 -0.0288  0.0962 -0.0522  0.0555 -0.0309 -0.1227  0.5745 -0.0114] b=0.0000 MSE=0.1
Epoch 80 → w=[ 0.2415 -0.0246  0.0919 -0.0456  0.0481 -0.0309 -0.1175  0.6217 -0.0125] b=0.0000 MSE=0.1
Epoch 100 → w=[ 0.2043 -0.0213  0.0895 -0.0421  0.0437 -0.0306 -0.1117  0.6645 -0.014 ] b=0.0000 MSE=0.1
Epoch 120 → w=[ 0.1682 -0.0183  0.0878 -0.0399  0.0405 -0.0302 -0.1059  0.7043 -0.0156] b=0.0000 MSE=0.0
Epoch 140 → w=[ 0.1338 -0.0157  0.0865 -0.0381  0.0378 -0.0298 -0.1003  0.7416 -0.0171] b=0.0000 MSE=0.0
Epoch 160 → w=[ 0.101  -0.0133  0.0853 -0.0366  0.0354 -0.0294 -0.095   0.7769 -0.0185] b=0.0000 MSE=0.0
Epoch 180 → w=[ 0.07   -0.0111  0.0842 -0.0352  0.0332 -0.029

In [43]:
def r2 (y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

train_r2 = r2(y_train_scaled, predict(X_train_scaled, w, b))
test_r2 = r2(y_test_scaled, predict(X_test_scaled, w, b))

train_mse = mse(y_train_scaled, predict(X_train_scaled, w, b))
test_mse = mse(y_test_scaled, predict(X_test_scaled, w, b))
print(f"Train MSE: {train_mse:.2f}")
print(f"Test MSE:  {test_mse:.2f}")
print(f"Train R²:  {train_r2:.4f}")
print(f"Test R²:   {test_r2:.4f}")

Train MSE: 0.02
Test MSE:  0.01
Train R²:  0.9769
Test R²:   0.9223


In [44]:
y_test_pred_scaled = predict(X_test_scaled, w, b)
y_test_pred = (y_test_pred_scaled * y_std) + y_mean
y_test_real = (y_test_scaled * y_std) + y_mean

test_mse_real = mse(y_test_real, y_test_pred)
rmse = np.sqrt(test_mse_real)
print(f"Typical error: ±{rmse:.2f} lakhs")

Typical error: ±0.67 lakhs


In [45]:
def predict_price(present_price, kms_driven, fuel_type, seller_type, transmission, owner, year):
    fuel_type_map = {'Petrol': 0, 'Diesel': 1, 'CNG': 2}
    seller_type_map = {'Dealer': 0, 'Individual': 1}
    transmission_map = {'Manual': 0, 'Automatic': 1}

    car_age = 2025 - year
    price_per_age = present_price / car_age
    kms_per_age = kms_driven / car_age
    features = np.array([[  
    present_price, kms_driven, fuel_type_map[fuel_type], 
    seller_type_map[seller_type], transmission_map[transmission], 
    owner, car_age, price_per_age, kms_per_age
   ]])
    features_scaled = (features - X_mean) / X_std
    predicted_price_scaled = predict(features_scaled, w, b)
    predicted_price = ((predicted_price_scaled * y_std) + y_mean)[0]
    return print(f"Estimated Price: {predicted_price - 0.67:.2f} – {predicted_price + 0.67:.2f} lakhs")



predict_price(
    present_price=5.59,
    kms_driven=27000,
    fuel_type='Petrol',
    seller_type='Dealer',
    transmission='Manual',
    owner=0,
    year=2014
)

Estimated Price: 2.69 – 4.03 lakhs


In [46]:
model_data = {
    'w': w,
    'b': b,
    'X_mean': X_mean,
    'X_std': X_std,
    'y_mean': y_mean,
    'y_std': y_std,
    'rmse': 0.67
}

with open('car_price_model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

In [47]:
df['price_ratio'] = df['Selling_Price'] / df['Present_Price']
print(df['price_ratio'].describe())
print(f"\nCars where selling > present: {(df['price_ratio'] > 1).sum()}")

count    301.000000
mean       0.634452
std        0.202369
min        0.105352
25%        0.505051
50%        0.654628
75%        0.790068
max        0.989255
Name: price_ratio, dtype: float64

Cars where selling > present: 0


In [52]:
print("Present Price stats:")
print(df['Present_Price'].describe())
print("\nYear stats:")
print(df['Car_Age'].describe())

Present Price stats:
count    301.000000
mean       7.628472
std        8.644115
min        0.320000
25%        1.200000
50%        6.400000
75%        9.900000
max       92.600000
Name: Present_Price, dtype: float64

Year stats:
count    301.000000
mean      11.372093
std        2.891554
min        7.000000
25%        9.000000
50%       11.000000
75%       13.000000
max       22.000000
Name: Car_Age, dtype: float64
